# 00 · Reset Rebuild Lakehouse — YelpConnect Trust & Safety

**Tipo:** Rebuild / Aceptación End-to-End  
**Naturaleza:** Destructivo sobre productos derivados  
**Ámbito:** Bronze · Silver · Gold · Gobierno operativo · Checkpoints

**Cierra con:** preparación del Lakehouse para una reconstrucción completa y reproducible desde RAW.

---

## Propósito

Restablecer los productos derivados del Lakehouse para permitir una reconstrucción completa y reproducible del pipeline **YelpConnect Trust & Safety**.

Este notebook es el punto inicial del:

`JOB_YELPCONNECT_TS_REBUILD_E2E`

y se utiliza exclusivamente para:

- rebuild completo;
- pruebas de aceptación;
- validación de reproducibilidad;
- reconstrucción controlada del Lakehouse.

> **Importante:** este notebook no debe formar parte del futuro pipeline operativo/incremental diario.

---

## Responsabilidades

Este notebook:

- elimina productos Gold derivados;
- elimina tablas Silver conformadas y derivadas;
- elimina tablas Bronze;
- elimina metadata operativa reconstruible de Data Quality;
- elimina metadata operativa reconstruible de Trust & Safety;
- elimina el checkpoint de Auto Loader utilizado por Review;
- conserva los archivos fuente RAW;
- conserva metadata maestra de gobierno.

Este notebook **NO**:

- transforma datos;
- ejecuta reglas Data Quality;
- ejecuta reglas Trust & Safety;
- calcula features;
- calcula `weighted_risk_score`;
- calcula `signal_level`;
- determina `requires_investigation`;
- determina prioridades P1 / P2 / P3;
- publica productos Gold.

---

## Principio de arquitectura

```text
RAW
 │
 │ se conserva
 ▼
Bronze
 │
 │ se reconstruye
 ▼
Silver
 │
 │ se reconstruye
 ▼
Trust & Safety
 │
 │ se reevalúa
 ▼
Gold

In [0]:
# COMMAND ----------

# ============================================================
# CONFIGURACIÓN DE CAPAS
# ============================================================
#
# Se centralizan los nombres de catálogo y schemas utilizados
# durante el proceso de reset.
#
# No se crean ni modifican schemas en este notebook.
# ============================================================

CATALOG = "workspace"

BRONZE = f"{CATALOG}.yelp_bronze"
SILVER = f"{CATALOG}.yelp_silver"
GOLD = f"{CATALOG}.yelp_gold"
GOV = f"{CATALOG}.yelp_gov"

print("=" * 72)
print("RESET YELPCONNECT TRUST & SAFETY")
print("=" * 72)


---

### Markdown 2 — Gold


## 1 · Reset de productos Gold

Gold representa la capa de publicación certificada para consumo analítico, investigación y visualización.

Durante un rebuild, estos productos deben eliminarse para garantizar que sean reconstruidos únicamente a partir de los resultados de la nueva ejecución.

### Se eliminan

- dimensiones;
- hechos;
- consolidación de riesgo;
- cola de investigación;
- KPIs Trust & Safety.

### Principio

Gold **publica productos certificados**.

Gold no debe recalcular:

- reglas Trust & Safety;
- `weighted_risk_score`;
- `requires_investigation`;
- `investigation_priority`.

> La reconstrucción posterior de Gold debe consumir los resultados producidos por las etapas previas de evaluación y consolidación T&S.

In [0]:
# COMMAND ----------

# ============================================================
# 1. GOLD
# ============================================================

gold_tables = [
    "gld_kpi_trust_safety_rule",
    "gld_kpi_trust_safety_overview",
    "gld_kpi_trust_safety_daily",
    "gld_kpi_trust_safety_business",
    "gld_investigation_queue",
    "gld_review_risk",
    "gld_fact_review_evaluation",
    "gld_dim_evaluation_run",
    "gld_dim_rule",
    "gld_dim_user",
    "gld_dim_business"
]

for table in gold_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {GOLD}.{table}"
    )
    print(f"✔ DROP {GOLD}.{table}")

# COMMAND ----------

## 2 · Reset de Silver

Silver contiene datos conformados y estructuras derivadas utilizadas por el proceso analítico.

Durante el rebuild se eliminan:

- entidades Silver conformadas;
- cuarentena;
- features Trust & Safety;
- evaluaciones de reglas;
- consolidación de riesgo;
- estructuras históricas o de compatibilidad aún presentes.

### Principio de arquitectura

Silver:

- tipa;
- normaliza;
- conforma;
- prepara los datos para consumo analítico.

Las estructuras Trust & Safety almacenadas en Silver representan resultados heurísticos.

> Una señal T&S **no confirma fraude**.  
> Representa una condición potencialmente sospechosa que puede requerir análisis posterior.

### Estructuras Trust & Safety

El rebuild elimina:

```text
slv_review_features
slv_review_evaluation
slv_review_risk

In [0]:
# COMMAND ----------

# ============================================================
# 2. SILVER
# ============================================================

silver_tables = [
    # Productos derivados Trust & Safety
    "slv_review_risk",
    "slv_review_evaluation",
    "slv_review_features",

    # Estructura histórica / obsoleta
    "slv_review_signal_summary",

    # Silver core
    "slv_review_cuarentena",
    "slv_checkin_event",
    "slv_tip",
    "slv_review",
    "slv_user",
    "slv_business"
]

for table in silver_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {SILVER}.{table}"
    )
    print(f"✔ DROP {SILVER}.{table}")

# COMMAND ----------


---

### Markdown 4 — Bronze

```markdown
## 3 · Reset de Bronze

Bronze materializa la ingesta de los archivos RAW.

Durante el rebuild se eliminan las tablas Delta Bronze, pero **no se eliminan los archivos fuente**.

### Principio de arquitectura

```text
RAW
 │
 │ fuente original
 ▼
Bronze
 │
 │ materialización de ingesta
 ▼
Silver

In [0]:
# COMMAND ----------

# ============================================================
# 3. BRONZE
# ============================================================

bronze_tables = [
    "brz_checkin",
    "brz_tip",
    "brz_review",
    "brz_user",
    "brz_business"
]

for table in bronze_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {BRONZE}.{table}"
    )
    print(f"✔ DROP {BRONZE}.{table}")

# COMMAND ----------


Esto está alineado con el comportamiento actual del notebook: elimina Bronze pero no RAW. 

---

### Markdown 5 — Gobierno operativo

```markdown
## 4 · Reset de gobierno operativo

No toda la metadata de gobierno tiene el mismo ciclo de vida.

Se distinguen dos categorías:

### Gobierno maestro

Representa información relativamente estable del dominio:

- glosario de negocio;
- diccionario de datos;
- matriz RACI.

### Gobierno operativo

Representa información relacionada con:

- reglas reconstruibles;
- ejecuciones;
- resultados de Data Quality;
- Evaluation Runs;
- evidencias generadas por el pipeline.

El gobierno operativo se elimina durante el rebuild porque debe regenerarse en la nueva ejecución.

---

### Data Quality

Se eliminan:

```text
dq_reglas
dq_resultados

In [0]:
# COMMAND ----------

gov_tables = [
    "evaluation_run",
    "evaluation_rule_version",
    "evaluation_rule",
    "dq_resultados",
    "dq_reglas"
]

for table in gov_tables:
    spark.sql(
        f"DROP TABLE IF EXISTS {GOV}.{table}"
    )
    print(f"✔ DROP {GOV}.{table}")

# COMMAND ----------



---

### Markdown 6 — Gobierno maestro

```markdown
## 5 · Gobierno maestro que se conserva

Los siguientes objetos no representan una ejecución específica del pipeline y por ello se preservan durante el rebuild.

| Objeto | Responsabilidad |
|---|---|
| `diccionario_datos` | Descripción y metadata de los elementos de datos |
| `glosario_negocio` | Definiciones semánticas del dominio |
| `matriz_raci` | Responsabilidades de gobierno |

Esta separación permite distinguir:

```text
Metadata maestra

In [0]:
# COMMAND ----------

print()
print("SE CONSERVAN:")
print("✔ workspace.yelp_gov.diccionario_datos")
print("✔ workspace.yelp_gov.glosario_negocio")
print("✔ workspace.yelp_gov.matriz_raci")

# COMMAND ----------


El notebook actual conserva precisamente `diccionario_datos`, `glosario_negocio` y `matriz_raci`.

---

### Markdown 7 — Checkpoint Review

```markdown
## 6 · Reset del checkpoint de Review

La ingesta de Review utiliza Auto Loader.

Auto Loader mantiene estado mediante un checkpoint para recordar qué archivos ya fueron procesados.

Durante una ejecución incremental normal, este comportamiento evita reprocesar archivos ya consumidos.

Durante un **rebuild completo**, en cambio, conservar ese checkpoint impediría volver a leer Review desde RAW.

Por esta razón se elimina:

```text
/Volumes/workspace/yelp_bronze/raw/_checkpoints/review

In [0]:
# COMMAND ----------

CHECKPOINT_REVIEW = (
    "/Volumes/workspace/yelp_bronze/raw/_checkpoints/review"
)

try:
    dbutils.fs.rm(
        CHECKPOINT_REVIEW,
        True
    )

    print(
        f"✔ Checkpoint eliminado: "
        f"{CHECKPOINT_REVIEW}"
    )

except Exception as exc:

    print(
        "⚠ No se encontró o no pudo eliminarse "
        "el checkpoint Review."
    )

    print(exc)

# COMMAND ----------


La ruta y la finalidad del checkpoint coinciden con el notebook actual. 

---

### Markdown 8 — Validación RAW

```markdown
## 7 · Validación de preservación RAW

Como control posterior al reset se inspecciona el Volume RAW.

Esta validación permite comprobar que:

- las tablas derivadas fueron eliminadas;
- los archivos fuente continúan disponibles;
- el Lakehouse conserva la información necesaria para reconstruirse;
- el proceso de reset no destruyó la fuente original.

### Ruta validada

```text
/Volumes/workspace/yelp_bronze/raw

In [0]:
# COMMAND ----------

RAW = (
    "/Volumes/workspace/yelp_bronze/raw"
)

print()
print("Contenido RAW conservado:")

display(
    dbutils.fs.ls(
        RAW
    )
)

# COMMAND ----------


---

### Markdown 9 — Resultado esperado

```markdown
## 8 · Estado esperado después del Reset

Al finalizar correctamente el notebook, el entorno debe quedar en el siguiente estado:

| Componente | Estado |
|---|---|
| RAW | Conservado |
| Bronze | Eliminado |
| Silver Core | Eliminado |
| Features T&S | Eliminado |
| Evaluaciones T&S | Eliminado |
| Risk Consolidation | Eliminado |
| Gold | Eliminado |
| Resultados DQ | Eliminado |
| Catálogo DQ operativo | Eliminado |
| Catálogo T&S operativo | Eliminado |
| Evaluation Run | Eliminado |
| Diccionario | Conservado |
| Glosario | Conservado |
| RACI | Conservado |
| Checkpoint Review | Eliminado |

El entorno queda preparado para iniciar una nueva reconstrucción controlada del Lakehouse.

In [0]:
# COMMAND ----------

print()
print("=" * 72)
print("RESET FINALIZADO")
print("=" * 72)

print()
print("Las tablas derivadas fueron eliminadas.")
print("Los archivos RAW fueron conservados.")
print()
print("Siguiente paso:")
print("01_configurar_lakehouse / catálogos / Bronze")
print("=" * 72)

# COMMAND ----------